# Sentiment Classification Using BERT

情緒分類使用BERT神經網路

    負面:0 正面:1

    負面:0 正面:1 中立:2

    負面:0 正面:1 中立:2 無情緒:3



# Load model and tokenizer

In [21]:
from transformers import AutoTokenizer, pipeline,BertForSequenceClassification
import torch

In [22]:
# Setting device on GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cpu


In [23]:
!pip install transformers

In [24]:
# You can download the best trained model from huggingface
# https://huggingface.co/clhuang

# (1) Load model from huggingface
# model = AutoModelForSequenceClassification.from_pretrained("clhuang/albert-sentiment")
model = BertForSequenceClassification.from_pretrained("clhuang/albert-sentiment", num_labels=2) # specify number of labels

# (2) or Load model from local
# best_model = "best-model-v1"  #
# # model = AutoModelForSequenceClassification.from_pretrained("./my-best-model").to(device)
# model = BertForSequenceClassification.from_pretrained(best_model, num_labels=2).to(device)  # specify number of labels


In [25]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 312, padding_idx=0)
      (position_embeddings): Embedding(512, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-3): 4 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=312, out_features=312, bias=True)
              (key): Linear(in_features=312, out_features=312, bias=True)
              (value): Linear(in_features=312, out_features=312, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=312, out_features=312, bias=True)
              (LayerNorm): LayerNorm((312,), eps=1e-1

In [26]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("clhuang/albert-sentiment")
model = AutoModelForSequenceClassification.from_pretrained("clhuang/albert-sentiment")

In [27]:
# tokenizer
# tokenizer = AutoTokenizer.from_pretrained(best_model) # or from local
tokenizer = AutoTokenizer.from_pretrained("clhuang/albert-sentiment") #from huggingface
# tokenizer = BertTokenizer.from_pretrained("bert-base-chinese") # or from hugginfacd bert-base-chinese

In [28]:
len(tokenizer)

21128

In [29]:
# tokenize can encode text to input_ids and decode input_ids to text
tokenizer.get_vocab()

{'522': 11510,
 '阀': 7322,
 'acg': 11553,
 'single': 12148,
 '##§': 13351,
 '##º': 13358,
 '##tee': 12729,
 '##筱': 18093,
 '##跄': 19703,
 '乓': 729,
 'pmid': 11804,
 '##剽': 14260,
 '##悠': 15698,
 '峇': 2280,
 '構': 3539,
 '樊': 3557,
 'shtml': 11501,
 '泉': 3787,
 'ping': 11891,
 '##いい': 12993,
 'niusnews': 13212,
 '##荡': 18839,
 '##责': 19626,
 '##陞': 20422,
 '##辣': 19850,
 '##que': 9477,
 '##├': 13584,
 '食': 7608,
 '跪': 6661,
 '307': 10486,
 '##肽': 18569,
 'dl': 11961,
 '咛': 1481,
 '咲': 1494,
 '##mah': 10394,
 '##肛': 18554,
 '監': 4675,
 '##沐': 16816,
 '迈': 6815,
 '##皆': 17696,
 '##訕': 19304,
 '##gio': 12676,
 '##蛇': 19083,
 '##阁': 20380,
 'rock': 10092,
 '##冾': 14168,
 '##点': 17214,
 '##啼': 14639,
 '312': 10578,
 '##在': 14819,
 '##癢': 17680,
 '##鯽': 20868,
 '怔': 2585,
 '修': 934,
 '靓': 7472,
 '79': 8428,
 '柄': 3376,
 'から': 8526,
 '##to': 8527,
 '438': 13174,
 '纬': 5281,
 '撂': 3047,
 '嗳': 1641,
 '5a': 10594,
 '焰': 4195,
 '萊': 5844,
 '##唾': 14609,
 '##墨': 14931,
 '##懿': 15817,
 '##⦿': 13644,


In [30]:
text="我喜歡"
# prepare our text into tokenized sequence
inputs = tokenizer(text)
inputs

{'input_ids': [101, 2769, 1599, 3631, 102], 'token_type_ids': [0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1]}

In [31]:
tokenizer.decode(inputs['input_ids'])

'[CLS] 我 喜 歡 [SEP]'

# Predict or generate result using pipeline

    可能的輸出結果如下:

    [{'label': 'LABEL_1', 'score': 0.9885562062263489}]
    [{'label': 'LABEL_0', 'score': 0.9052111506462097}]

    因此需要用到if去判端label的值，才能決定score是正面還是負面。
    若為:LABEL_1 就是正面的score
    若為:LABEL_0 就是負面的score

In [32]:
sentiment_classify = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer)

Device set to use cpu


In [33]:
new_text = '速度很快，昨天下單，今天上午就到啦，看著挺不錯。'
sentiment_classify(new_text)

[{'label': 'LABEL_1', 'score': 0.9885562062263489}]

In [34]:
outputs = sentiment_classify(new_text)
outputs[0]['score']

0.9885562062263489

In [35]:
type(outputs[0]['score'])

float

In [36]:
# Positive probability
round(outputs[0]['score'],2)

0.99

In [37]:
# Negative probability
round(1 - round(outputs[0]['score'],2),2)

0.01

In [38]:
new_text = '不喜歡這款產品'
sentiment_classify(new_text)

[{'label': 'LABEL_0', 'score': 0.9052111506462097}]

# Define prediction function using pipeline

In [39]:
def get_sentiment_proba(text):
    max_length = 300 # 最多字數 若超出模型訓練時的字數，以模型最大字數為依據
    #max_length = 512 # 最多字數 若超出模型訓練時的字數，以模型最大字數為依據
    outputs = sentiment_classify(text, padding=True, max_length=max_length, truncation=True)
    if outputs[0]['label']=='LABEL_1':
        # Get the positive score
        prob_positive = round(outputs[0]['score'],2)
        prob_negatitive = round(1 - prob_positive, 2)
    else:
        # Calculate the negative score
        prob_negatitive = round(outputs[0]['score'],2)
        prob_positive = round(1 - prob_negatitive, 2)

    response = {'Negative':prob_negatitive, 'Positive': prob_positive}
    return response

In [40]:
new_text = '速度很快，昨天下單，今天上午就到啦，看著挺不錯。'
get_sentiment_proba( new_text )

{'Negative': 0.01, 'Positive': 0.99}

In [41]:
new_text = '已經買了這種蘋果好多次了，寶寶喜歡上了這款蘋果，一直選擇這款'
get_sentiment_proba( new_text )

{'Negative': 0.02, 'Positive': 0.98}

In [42]:
new_text = '不喜歡這款產品'

get_sentiment_proba( new_text )

{'Negative': 0.91, 'Positive': 0.09}

In [43]:
new_text = '非常不喜歡這款產品'

get_sentiment_proba( new_text )

{'Negative': 0.72, 'Positive': 0.28}

# Define prediction function using model or model.generate()

In [44]:
## Pediction
target_names=['Negative','Positive']
max_length = 200 # 最多字數 若超出模型訓練時的字數，以模型最大字數為依據
def get_sentiment_proba_from_model(text):
    # prepare our text into tokenized sequence
    inputs = tokenizer(text, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)
    # perform inference to our model
    outputs = model(**inputs)
    # get output probabilities by doing softmax
    probs = outputs[0].softmax(1)

    response = {'Negative': round(float(probs[0, 0]), 2), 'Positive': round(float(probs[0, 1]), 2)}
    # executing argmax function to get the candidate label
    #return probs.argmax()
    return response

In [45]:
new_text = '不喜歡這款產品'

get_sentiment_proba_from_model( new_text )

{'Negative': 0.91, 'Positive': 0.09}

# Pediction模型使用

## label <--> id

In [46]:
# Map labels to integers
categories=['負面','正面']

In [47]:

label_to_id = { cate : i for i, cate in enumerate(categories)}

In [48]:
label_to_id

{'負面': 0, '正面': 1}

In [49]:
id_to_label = { i : cate for i, cate in enumerate(categories)}

In [50]:
id_to_label

{0: '負面', 1: '正面'}

In [51]:

# Function to make predictions
def predict_sentiment(text, model, tokenizer, device):
    max_length = 512 # 最多字數 若超出模型訓練時的字數，以模型最大字數為依據
    # Tokenize the input text
    inputs = tokenizer(
        text,
        max_length=max_length,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # Get model predictions
    with torch.no_grad():
        outputs = model(**inputs)

    # Extract logits and apply softmax to get probabilities
    # logits = outputs.logits
    logits = outputs["logits"]  # 取出 logits


    probabilities = torch.nn.functional.softmax(logits, dim=-1)

    # Get the predicted class (0: negative, 1: positive)
    predicted_class = torch.argmax(probabilities, dim=-1).item()

    # Get the class name using id_to_label
    predicted_label = id_to_label[predicted_class]

    # Get the confidence score
    confidence = probabilities[0][predicted_class].item()

    return {
        "text": text,
        "sentiment": predicted_label,
        "confidence": round(confidence,2),
        "probabilities": {
            id_to_label[i]: round(prob.item(),2) for i, prob in enumerate(probabilities[0])
        }
    }


In [52]:
text = "今天天氣真好，我很開心"
predict_sentiment(text, model, tokenizer, device)

{'text': '今天天氣真好，我很開心',
 'sentiment': '正面',
 'confidence': 0.81,
 'probabilities': {'負面': 0.19, '正面': 0.81}}

In [53]:
text = "這個產品品質差，服務更糟糕"
predict_sentiment(text, model, tokenizer, device)

{'text': '這個產品品質差，服務更糟糕',
 'sentiment': '負面',
 'confidence': 0.99,
 'probabilities': {'負面': 0.99, '正面': 0.01}}

In [54]:
text = "這家餐廳的食物美味，環境也很舒適"
predict_sentiment(text, model, tokenizer, device)

{'text': '這家餐廳的食物美味，環境也很舒適',
 'sentiment': '正面',
 'confidence': 0.98,
 'probabilities': {'負面': 0.02, '正面': 0.98}}